# VascularAge — Phase 5 / S2 Robustness

This notebook is the execution shell for the **prospectively locked S2 robustness endpoint**. It does not redefine the scientific operators.

Phase-5 S2 lock: `97d831b3b3770dddcec9101f3095ecd92d190c4c9f4a1c17e67c2a6c3b8dfb05`

Use **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. The run verifies the preserved Phase-4 A001 evidence bundle before computing L1/L∞ robustness.


In [1]:
from pathlib import Path
import json, os, subprocess, sys
from google.colab import drive

BRANCH = "phase-05-s2-robustness"
S2_LOCK = "97d831b3b3770dddcec9101f3095ecd92d190c4c9f4a1c17e67c2a6c3b8dfb05"
VQ_SHA = "79891036e61df3096536da8f647f2297b0d88252"
drive.mount("/content/drive")

os.environ["XDG_DATA_HOME"] = "/content/drive/MyDrive/VascularAge/phase_01/xdg/data"
os.environ["XDG_CACHE_HOME"] = "/content/drive/MyDrive/VascularAge/phase_01/xdg/cache"
os.environ["XDG_STATE_HOME"] = "/content/drive/MyDrive/VascularAge/phase_01/xdg/state"

REPO = Path("/content/VascularAge")
VQ = Path("/content/VascuQuest")
subprocess.run(["rm", "-rf", str(REPO), str(VQ)], check=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, "https://github.com/khalid-saqr/VascularAge.git", str(REPO)], check=True)
subprocess.run(["git", "clone", "https://github.com/KNOWDYN/VascuQuest.git", str(VQ)], check=True)
subprocess.run(["git", "checkout", VQ_SHA], cwd=VQ, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(VQ)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test]"], check=True)

va_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
vq_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=VQ, text=True).strip()
manifest = json.loads((REPO / "phase5" / "S2_LOCK.json").read_text())
assert manifest["s2_lock_sha256"] == S2_LOCK
assert vq_commit == VQ_SHA
print("VascularAge:", va_commit)
print("VascuQuest:", vq_commit)
print("Phase-5 S2 lock:", S2_LOCK)

import jax
print("JAX backend:", jax.default_backend())
print("JAX devices:", [str(d) for d in jax.devices()])
assert jax.default_backend() != "cpu", "Select a GPU runtime before executing Phase 5"


Mounted at /content/drive
VascularAge: b45ee69c423782aac06ef35391f9ccde1a23fe8f
VascuQuest: 79891036e61df3096536da8f647f2297b0d88252
Phase-5 S2 lock: 97d831b3b3770dddcec9101f3095ecd92d190c4c9f4a1c17e67c2a6c3b8dfb05
JAX backend: gpu
JAX devices: ['cuda:0']


In [2]:
cmd = [
    sys.executable,
    str(REPO / "scripts" / "phase5_colab.py"),
    "--execute-s2",
    "--expected-s2-lock-sha", S2_LOCK,
    "--phase4-root", "/content/drive/MyDrive/VascularAge/phase_04/locked_trial_amendment001_20260829T153743Z",
    "--output-parent", "/content/drive/MyDrive/VascularAge/phase_05",
    "--repo-root", str(REPO),
    "--vq-root", str(VQ),
]
proc = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr, file=sys.stderr)
if proc.returncode != 0:
    raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr)
assert "PHASE 5 S2 EXTERNAL EXECUTION: COMPLETE" in proc.stdout
print("PHASE 5 S2 NOTEBOOK: SUCCESS")


.........................                                                [100%]

JAX backend: gpu
JAX devices: ['cuda:0']
VERIFIED model_variations /content/drive/MyDrive/VascularAge/phase_01/xdg/data/VascuQuest/source/pwdb_model_variations.csv
VERIFIED common_site_waveforms_csv /content/drive/MyDrive/VascularAge/phase_01/xdg/data/VascuQuest/source/PWs_csv.zip
Executing: /usr/bin/python3 /content/VascularAge/scripts/phase5_execute_s2.py --execute-s2 --expected-s2-lock-sha 97d831b3b3770dddcec9101f3095ecd92d190c4c9f4a1c17e67c2a6c3b8dfb05 --repo-root /content/VascularAge --pwdb-root /content/drive/MyDrive/VascularAge/phase_01/xdg/data/VascuQuest/source --phase4-evidence-root /content/drive/MyDrive/VascularAge/phase_04/locked_trial_amendment001_20260829T153743Z --output-root /content/drive/MyDrive/VascularAge/phase_05/locked_trial_phase5_s2_20260829T162522Z
S2 block 25 vs 35
S2 block 25 vs 45
S2 block 25 vs 55
S2 block 25 vs 65
S2 block 25 vs 75
S2 block 35 vs 45
S2 block 35 vs 55
S2 block